We use the newly cleaned upgraded feature dataset to train and evaluate the model

In [94]:
import pandas as pd
from pathlib import Path
import sklearn as sk

In [95]:
data_dir = Path.cwd().joinpath("swiss-bank-risk-project", "data")

processed_data_folder = data_dir.joinpath("processed")
processed_data_folder.mkdir(parents=True, exist_ok=True)

feature_weekly_data = processed_data_folder.joinpath("feature_weekly_data.csv")

feature_weekly_df = pd.read_csv(feature_weekly_data, parse_dates=["date"])
#display(feature_weekly_df.head())

In [96]:
feature_weekly_df = feature_weekly_df[(feature_weekly_df["date"] >= "2015-01-01") & (feature_weekly_df["date"] <= "2025-07-31")].copy()

feature_weekly_df = feature_weekly_df.sort_values("date").reset_index(drop=True) 
# We want observations to be in chronological order

print("Shape:")
print(feature_weekly_df.shape)


print("Date range:")
print(feature_weekly_df["date"].min())
print(feature_weekly_df["date"].max())


print("Missing values:")
print(feature_weekly_df.isna().sum())


print("First rows:")
display(feature_weekly_df.head())

Shape:
(552, 19)
Date range:
2015-01-02 00:00:00
2025-07-25 00:00:00
Missing values:
date                                     0
ciss_index                               0
chf-eur_exchange_rate_control            0
commercial_bank_bond_yield               0
government_bond_yield                    0
commercial_bank_bond_spread              0
snb_policy_rate                          0
ciss_change                              0
bond_spread_change                       0
commercial_bond_yield_change             0
government_bond_yield_change             0
policy_rate_change                       0
ciss_volatility_past_4_weeks             0
bond_spread_volatility_past_4_weeks      0
exchange_rate_volatility_past_4_weeks    0
ciss_lag_1w                              0
ciss_lag_4w                              0
bond_spread_lag_1w                       0
bond_spread_lag_4w                       0
dtype: int64
First rows:


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w
0,2015-01-02,0.007444,1.20255,0.4535,0.2280,0.2255,-0.75,-0.009657,-0.0230,-0.0115,0.0115,-0.5,0.006559,0.039160,0.000521,0.017100,0.007930,0.2485,0.3358
1,2015-01-09,0.025402,1.20122,0.3630,0.1078,0.2552,-0.75,0.017958,0.0297,-0.0905,-0.1202,0.0,0.008001,0.020257,0.000781,0.007444,0.013188,0.2255,0.3168
2,2015-01-16,0.028616,1.12876,0.2646,0.0142,0.2504,-0.75,0.003214,-0.0048,-0.0984,-0.0936,0.0,0.009469,0.013237,0.036768,0.025402,0.023025,0.2552,0.2746
3,2015-01-23,0.062939,0.99926,0.0048,-0.3500,0.3548,-0.75,0.034324,0.1044,-0.2598,-0.3642,0.0,0.023181,0.057055,0.095561,0.028616,0.017100,0.2504,0.2485
4,2015-01-30,0.117127,1.02532,0.1148,-0.2908,0.4056,-0.75,0.054188,0.0508,0.1100,0.0592,0.0,0.042605,0.076447,0.093599,0.062939,0.007444,0.3548,0.2255


In [97]:
four_week_forward_commercial_bank_bond_spread = feature_weekly_df["commercial_bank_bond_spread"].shift(-4)

feature_weekly_df.insert(19, "four_week_forward_commercial_bank_bond_spread", four_week_forward_commercial_bank_bond_spread)

four_week_forward_commercial_bank_bond_spread_change = feature_weekly_df["four_week_forward_commercial_bank_bond_spread"] - feature_weekly_df["commercial_bank_bond_spread"]
feature_weekly_df.insert(20, "four_week_forward_commercial_bank_bond_spread_change", four_week_forward_commercial_bank_bond_spread_change)
display(feature_weekly_df.head())
display(feature_weekly_df.tail(4))


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,...,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w,four_week_forward_commercial_bank_bond_spread,four_week_forward_commercial_bank_bond_spread_change
0,2015-01-02,0.007444,1.20255,0.4535,0.2280,0.2255,-0.75,-0.009657,-0.0230,-0.0115,...,-0.5,0.006559,0.039160,0.000521,0.017100,0.007930,0.2485,0.3358,0.4056,0.1801
1,2015-01-09,0.025402,1.20122,0.3630,0.1078,0.2552,-0.75,0.017958,0.0297,-0.0905,...,0.0,0.008001,0.020257,0.000781,0.007444,0.013188,0.2255,0.3168,0.4162,0.1610
2,2015-01-16,0.028616,1.12876,0.2646,0.0142,0.2504,-0.75,0.003214,-0.0048,-0.0984,...,0.0,0.009469,0.013237,0.036768,0.025402,0.023025,0.2552,0.2746,0.4018,0.1514
3,2015-01-23,0.062939,0.99926,0.0048,-0.3500,0.3548,-0.75,0.034324,0.1044,-0.2598,...,0.0,0.023181,0.057055,0.095561,0.028616,0.017100,0.2504,0.2485,0.4004,0.0456
4,2015-01-30,0.117127,1.02532,0.1148,-0.2908,0.4056,-0.75,0.054188,0.0508,0.1100,...,0.0,0.042605,0.076447,0.093599,0.062939,0.007444,0.3548,0.2255,0.4370,0.0314


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,...,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w,four_week_forward_commercial_bank_bond_spread,four_week_forward_commercial_bank_bond_spread_change
548,2025-07-04,0.030363,0.93396,0.9386,0.2976,0.6410,0.0,0.006354,0.0020,0.0152,...,0.0,0.003906,0.008776,0.002678,0.024009,0.015220,0.6390,0.62660,NaN,NaN
549,2025-07-11,0.028096,0.93320,0.9658,0.3344,0.6314,0.0,-0.002266,-0.0096,0.0272,...,0.0,0.002924,0.008711,0.003213,0.030363,0.022962,0.6410,0.63725,NaN,NaN
550,2025-07-18,0.030874,0.93158,0.9958,0.3546,0.6412,0.0,0.002778,0.0098,0.0300,...,0.0,0.003127,0.004608,0.002481,0.028096,0.030051,0.6314,0.62180,NaN,NaN
551,2025-07-25,0.032790,0.93282,0.9622,0.3262,0.6360,0.0,0.001916,-0.0052,-0.0336,...,0.0,0.001930,0.004668,0.000994,0.030874,0.024009,0.6412,0.63900,NaN,NaN


In [98]:
testing_start_date = pd.Timestamp("2024-01-01")

training_end_date = testing_start_date - pd.Timedelta(weeks=4)
# we end it 4 weeks prematurely because we have to compute four_week_forward_commercial_bank_bond_spread_change
# We don't want the data to overlap with that of the testing period.

train_df = feature_weekly_df[feature_weekly_df["date"] < training_end_date].copy()

test_df = feature_weekly_df[feature_weekly_df["date"] >= testing_start_date].copy()

print("Training period:")
print(train_df["date"].min())
print(train_df["date"].max())
print("Training shape:", train_df.shape)


print("Test period:")
print(test_df["date"].min())
print(test_df["date"].max())
print("Test shape:", test_df.shape)

#display(train_df)

Training period:
2015-01-02 00:00:00
2023-12-01 00:00:00
Training shape: (466, 21)
Test period:
2024-01-05 00:00:00
2025-07-25 00:00:00
Test shape: (82, 21)


In [99]:
spread_change_threshold = train_df["four_week_forward_commercial_bank_bond_spread_change"].quantile(0.8)
print(spread_change_threshold)

train_df[train_df["four_week_forward_commercial_bank_bond_spread_change"] > spread_change_threshold].shape


0.04075000000000001


(93, 21)

In [112]:
print(train_df["four_week_forward_commercial_bank_bond_spread_change"].isna().sum())

print(test_df["four_week_forward_commercial_bank_bond_spread_change"].isna().sum())


# Remove rows where the 4-week-forward spread change cannot be calculated
train_df = train_df.dropna(subset=["four_week_forward_commercial_bank_bond_spread_change"]).copy()

test_df = test_df.dropna(subset=["four_week_forward_commercial_bank_bond_spread_change"]).copy()


train_df["unusual_spread_widening"] = (train_df["four_week_forward_commercial_bank_bond_spread_change"]> spread_change_threshold).astype(int)
# checks if it's wider than the unsusual spread widening. If it's the case,they become values.
# 1 If it's true, 0 if it's false.

test_df["unusual_spread_widening"] = (test_df["four_week_forward_commercial_bank_bond_spread_change"]> spread_change_threshold).astype(int)



print("Training target:")
print(train_df["unusual_spread_widening"].value_counts())
# counts how many 0s and how many 1s there are in the target column.
print(train_df["unusual_spread_widening"].value_counts(normalize=True) * 100)
# counts how many 0s and how many 1s there are in the target column as a %


print("Testing target:")
print(test_df["unusual_spread_widening"].value_counts())
print(test_df["unusual_spread_widening"].value_counts(normalize=True) * 100)

0
0
Training target:
unusual_spread_widening
0    373
1     93
Name: count, dtype: int64
unusual_spread_widening
0    80.042918
1    19.957082
Name: proportion, dtype: float64
Testing target:
unusual_spread_widening
0    72
1     6
Name: count, dtype: int64
unusual_spread_widening
0    92.307692
1     7.692308
Name: proportion, dtype: float64


In [120]:
train_df["covid_dummy"] = ((train_df["date"] >= "2020-03-16") & (train_df["date"] <= "2020-06-22")).astype(int)
train_df["credit_suisse_dummy"] = ((train_df["date"] >= "2023-03-15") & (train_df["date"] <= "2023-06-12")).astype(int)
train_df["ukraine_war_dummy"] = ((train_df["date"] >= "2022-02-24") & (train_df["date"] <= "2022-06-30")).astype(int)
train_df["snb_floor_dummy"] = ((train_df["date"] >= "2015-01-15") & (train_df["date"] <= "2015-02-27")).astype(int)

print(train_df["covid_dummy"].value_counts())
print(train_df["credit_suisse_dummy"].value_counts())
print(train_df["ukraine_war_dummy"].value_counts())
print(train_df["snb_floor_dummy"].value_counts())

display(train_df[(train_df["date"] >= "2020-03-01") & (train_df["date"] <= "2020-06-30")])

test_df["covid_dummy"] = ((test_df["date"] >= "2020-03-16") & (test_df["date"] <= "2020-06-22")).astype(int)
test_df["credit_suisse_dummy"] = ((test_df["date"] >= "2023-03-15") & (test_df["date"] <= "2023-06-12")).astype(int)
test_df["ukraine_war_dummy"] = ((test_df["date"] >= "2022-02-24") & (test_df["date"] <= "2022-06-30")).astype(int)
test_df["snb_floor_dummy"] = ((test_df["date"] >= "2015-01-15") & (test_df["date"] <= "2015-02-27")).astype(int)

print(test_df["covid_dummy"].value_counts())
print(test_df["credit_suisse_dummy"].value_counts())
print(test_df["ukraine_war_dummy"].value_counts())
print(test_df["snb_floor_dummy"].value_counts())

covid_dummy
0    452
1     14
Name: count, dtype: int64
credit_suisse_dummy
0    453
1     13
Name: count, dtype: int64
ukraine_war_dummy
0    448
1     18
Name: count, dtype: int64
snb_floor_dummy
0    459
1      7
Name: count, dtype: int64


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,...,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w,four_week_forward_commercial_bank_bond_spread,four_week_forward_commercial_bank_bond_spread_change,unusual_spread_widening,covid_dummy,credit_suisse_dummy,ukraine_war_dummy,snb_floor_dummy
270,2020-03-06,0.264207,1.064000,-0.40220,-0.86640,0.4642,-0.75,0.179120,0.0060,-0.00580,...,0.028082,0.4582,0.4834,0.6158,0.1516,1,0,0,0,0
271,2020-03-13,0.480712,1.058720,-0.31920,-0.85580,0.5366,-0.75,0.216505,0.0724,0.08300,...,0.018316,0.4642,0.4734,0.6085,0.0719,1,0,0,0,0
272,2020-03-20,0.635157,1.054680,0.12440,-0.51440,0.6388,-0.75,0.154446,0.1022,0.44360,...,0.013567,0.5366,0.4562,0.6530,0.0142,0,1,0,0,0
273,2020-03-27,0.668939,1.059580,0.26940,-0.40800,0.6774,-0.75,0.033781,0.0386,0.14500,...,0.085087,0.6388,0.4582,0.6804,0.0030,0,1,0,0,0
274,2020-04-03,0.683114,1.056360,0.19520,-0.42060,0.6158,-0.75,0.014175,-0.0616,-0.07420,...,0.264207,0.6774,0.4642,0.6700,0.0542,1,1,0,0,0
275,2020-04-10,0.613669,1.056425,0.25675,-0.35175,0.6085,-0.75,-0.069445,-0.0073,0.06155,...,0.480712,0.6158,0.5366,0.6610,0.0525,1,1,0,0,0
276,2020-04-17,0.524237,1.052700,0.20875,-0.44425,0.6530,-0.75,-0.089431,0.0445,-0.04800,...,0.635157,0.6085,0.6388,0.6066,-0.0464,0,1,0,0,0
277,2020-04-24,0.469194,1.051880,0.18640,-0.49400,0.6804,-0.75,-0.055043,0.0274,-0.02235,...,0.668939,0.6530,0.6774,0.6205,-0.0599,0,1,0,0,0
278,2020-05-01,0.402017,1.056800,0.13925,-0.53075,0.6700,-0.75,-0.067177,-0.0104,-0.04715,...,0.683114,0.6804,0.6158,0.6192,-0.0508,0,1,0,0,0
279,2020-05-08,0.364588,1.053240,0.08420,-0.57680,0.6610,-0.75,-0.037429,-0.0090,-0.05505,...,0.613669,0.6700,0.6085,0.5830,-0.0780,0,1,0,0,0


covid_dummy
0    78
Name: count, dtype: int64
credit_suisse_dummy
0    78
Name: count, dtype: int64
ukraine_war_dummy
0    78
Name: count, dtype: int64
snb_floor_dummy
0    78
Name: count, dtype: int64


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,...,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w,four_week_forward_commercial_bank_bond_spread,four_week_forward_commercial_bank_bond_spread_change,unusual_spread_widening,covid_dummy,credit_suisse_dummy,ukraine_war_dummy,snb_floor_dummy


In [121]:
y_train = train_df["unusual_spread_widening"]
y_test = test_df["unusual_spread_widening"]

# One bracket returns a series while 2 brackets returns a df with just one column
X1_train = train_df[["ciss_change"]]
X1_test = test_df[["ciss_change"]]


X2_train = train_df[["ciss_change","chf-eur_exchange_rate_control","snb_policy_rate"]]

X2_test = test_df[["ciss_change", "chf-eur_exchange_rate_control", "snb_policy_rate"]]


X3_train = train_df[["ciss_change", "chf-eur_exchange_rate_control", "snb_policy_rate", "covid_dummy", "ukraine_war_dummy",
     "snb_floor_dummy","credit_suisse_dummy"]]

X3_test = test_df[["ciss_change", "chf-eur_exchange_rate_control", "snb_policy_rate", "covid_dummy", "ukraine_war_dummy",
     "snb_floor_dummy","credit_suisse_dummy"]]

if x1_train.shape == x2_train.shape == x3_train.shape == y_train.shape:
    print("The predictors and target have the same number of observations")

NameError: name 'x1_train' is not defined